# MS-TBD on Elliptic++ — bipartite vs. bipartite + tx-tx

End-to-end runnable notebook for **Multi-Scale Temporal Bipartite Diffusion** on the Elliptic++ transaction-classification task. We evaluate two graph variants under strict causal evaluation:

1. **V1: bipartite addr↔tx only** — wallets carry messages between transactions across timesteps via the bipartite edges.
2. **V2: bipartite addr↔tx + tx-tx graph** — adds the within-timestep `txs_edgelist.csv` edges by encoding each tx-tx edge `(T1, T2, t)` as two bipartite edges `(w_virtual, T1, t)` and `(w_virtual, T2, t)` through a unique virtual wallet. The same MS-TBD model handles both versions without modification.

**No wallet-wallet (`AddrAddr_edgelist.txt`) edges** are loaded — they have no timestamps, which would leak future structure under the causal protocol.

## Causality contract

For target tx `T` at time `t`, every input uses only data with `τ ≤ t`:
- **Edge weights** in MS-TBD: `w_e = 1[τ_e ≤ t] · exp(-β · (t - τ_e))`. Future edges (`τ_e > t`) get hard zero weight by construction.
- **Per-tx features `X`**: anchored to each tx's own timestep (no future tx features used for any T at time `t`).
- **Standardiser**: fit on TRAIN-ONLY rows (`t ≤ 29`), applied to all.
- `wallets_features.txt` is NOT loaded (lifetime aggregates would leak).

## Data split

- Train: labelled txs with `t ≤ 29`
- Val:   labelled txs with `30 ≤ t ≤ 34` (temporal val for best-epoch checkpointing)
- Test:  labelled txs with `t ≥ 35` (paper protocol test set; identical to phase 1)

## Comparison ablations

| | Description | Edges |
|---|---|---|
| **RF[raw 182]** | Phase-1 baseline | n/a |
| **MS-TBD V1** | bipartite addr↔tx only | 1.27M edges |
| **MS-TBD V2** | bipartite + tx-tx via virtual wallets | 1.27M + 2 × 234K = 1.74M edges |

The RF baseline is included so the MS-TBD numbers are directly comparable to your Phase 1 / 1c results.


In [19]:
!pip install -q \
  "numpy<2" \
  "torch-geometric>=2.4.0" \
  "scikit-learn>=1.3.0" \
  "pandas>=2.0.0" \
  "matplotlib>=3.7.0" \
  "seaborn>=0.12.0" \
  "pyyaml>=6.0" \
  "tqdm>=4.65.0"


In [ ]:
!pip install -q --force-reinstall \
  "numpy<2" \
  "pandas>=2.0,<2.2.3" \
  "scikit-learn>=1.3.0" \
  "torch-geometric>=2.4.0"

import os
os.kill(os.getpid(), 9)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.3.0 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 

In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    classification_report, precision_recall_curve,
)
import torch
import torch.nn as nn
import torch.nn.functional as F

# Walk up from CWD to find the repo root
# Mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ROOT = Path("/content/drive/MyDrive/stat175-final-project")

WALLETS_DIR      = ROOT / "actor_data"
TRANSACTIONS_DIR = ROOT / "data"
assert WALLETS_DIR.exists() and TRANSACTIONS_DIR.exists()
print(f"repo root: {ROOT}")

# === Data split (matches phase 1 test set; carves temporal val from train) ===
TRAIN_END     = 29       # MS-TBD train: t in [1, 29]
VAL_END       = 34       # MS-TBD val:   t in [30, 34]   (temporal — close to test in time)
TEST_START    = 35       # MS-TBD test:  t in [35, 49]   (identical to phase 1)
RF_TRAIN_END  = 34       # RF baseline trains on full t<=34 (matches phase 1)
N_TIME_STEPS  = 49
RANDOM_SEED   = 175
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# === MS-TBD hyperparameters ===
N_SCALES   = 5
N_ITERS    = 8
HIDDEN     = 64
DROPOUT    = 0.3
LR         = 3e-4
N_EPOCHS   = 50        # cap to keep CPU wall clock manageable
CLASS_WEIGHT_NEURAL = 5.0
GRAD_CLIP  = 5.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")


Mounted at /content/drive
repo root: /content/drive/MyDrive/stat175-final-project
device: cuda


In [2]:
print("Loading transactions...")
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.csv")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.csv")
tx_classes_df["label"] = tx_classes_df["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
tx_id_to_idx = {tid: i for i, tid in enumerate(tx_features_df["txId"].values)}
N_TX = len(tx_features_df)
tx_feat_cols = [c for c in tx_features_df.columns if c not in ("txId", "Time step")]
F_TX = len(tx_feat_cols)
merged = tx_features_df[["txId","Time step"]].merge(tx_classes_df[["txId","label"]], on="txId", how="left")
tx_time_np = merged["Time step"].astype(np.int64).values
tx_label   = merged["label"].fillna(-1).astype(np.int64).values
tx_X_raw   = tx_features_df[tx_feat_cols].fillna(0).astype(np.float32).values
print(f"  N_TX={N_TX:,}  F_TX={F_TX}")
print(f"  labels: illicit={int((tx_label==1).sum()):,}  licit={int((tx_label==0).sum()):,}  unknown={int((tx_label==-1).sum()):,}")

print("\nLoading actor bipartite edges (addr<->tx)...")
addr_tx_df = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")    # input wallet -> tx
tx_addr_df = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")    # tx -> output wallet
wallet_addrs = sorted(set(addr_tx_df["input_address"].unique()) | set(tx_addr_df["output_address"].unique()))
wallet_addr_to_idx = {a: i for i, a in enumerate(wallet_addrs)}
N_W = len(wallet_addr_to_idx)
print(f"  unique wallets in addr<->tx edges: {N_W:,}")

print("\nLoading tx<->tx edges (within-timestep)...")
tx_tx_df = pd.read_csv(TRANSACTIONS_DIR / "txs_edgelist.csv")
print(f"  raw tx-tx edges: {len(tx_tx_df):,}")


Loading transactions...
  N_TX=203,769  F_TX=182
  labels: illicit=4,545  licit=42,019  unknown=157,205

Loading actor bipartite edges (addr<->tx)...
  unique wallets in addr<->tx edges: 822,942

Loading tx<->tx edges (within-timestep)...
  raw tx-tx edges: 234,355


In [3]:
# === Build bipartite addr<->tx edge arrays ===
def map_bipartite(edge_df, addr_col, tx_col):
    df = edge_df[[addr_col, tx_col]].copy()
    df["w"]  = df[addr_col].map(wallet_addr_to_idx)
    df["tx"] = df[tx_col].map(tx_id_to_idx)
    df = df.dropna(subset=["w", "tx"])
    df["w"]  = df["w"].astype(np.int64)
    df["tx"] = df["tx"].astype(np.int64)
    return df["w"].values, df["tx"].values

w1, tx1 = map_bipartite(addr_tx_df, "input_address",  "txId")    # input wallet -> tx
w2, tx2 = map_bipartite(tx_addr_df, "output_address", "txId")    # tx -> output wallet (undirected, same encoding)

# Concatenate and dedupe (a wallet can be both input and output of the same tx)
row_w_all  = np.concatenate([w1,  w2])
col_tx_all = np.concatenate([tx1, tx2])
e_time_all = tx_time_np[col_tx_all]

# Dedupe (w, tx) pairs (preserves the first occurrence's timestep, which is the same anyway)
edge_df = pd.DataFrame({"w": row_w_all, "tx": col_tx_all, "t": e_time_all}).drop_duplicates(subset=["w","tx"])
row_w_bp  = edge_df["w"].values.astype(np.int64)
col_tx_bp = edge_df["tx"].values.astype(np.int64)
e_time_bp = edge_df["t"].values.astype(np.int64)
print(f"V1 (bipartite only):     {len(row_w_bp):,} edges over {N_W:,} wallets + {N_TX:,} txs")

# === Build V2 edges: bipartite + tx-tx via virtual wallets ===
# Each tx-tx edge (T1, T2, t_tt) becomes two bipartite edges:
#   (virtual_wallet_i, T1, t_tt)
#   (virtual_wallet_i, T2, t_tt)
# This makes the diffusion's tx -> wallet -> tx half-steps propagate score directly between
# tx pairs connected by tx-tx edges, while preserving the same MS-TBD bipartite operator.

tx_tx_df_local = tx_tx_df.copy()
tx_tx_df_local["t1"] = tx_tx_df_local["txId1"].map(tx_id_to_idx)
tx_tx_df_local["t2"] = tx_tx_df_local["txId2"].map(tx_id_to_idx)
tx_tx_df_local = tx_tx_df_local.dropna(subset=["t1","t2"])
tx_tx_df_local["t1"] = tx_tx_df_local["t1"].astype(np.int64)
tx_tx_df_local["t2"] = tx_tx_df_local["t2"].astype(np.int64)
# Within-timestep check (all Elliptic tx-tx edges are within-timestep — verified earlier)
tt_t1 = tx_tx_df_local["t1"].values
tt_t2 = tx_tx_df_local["t2"].values
tt_time_a = tx_time_np[tt_t1]
tt_time_b = tx_time_np[tt_t2]
assert (tt_time_a == tt_time_b).all(), "tx-tx edges expected within-timestep"
tt_time = tt_time_a   # = tt_time_b
n_tt = len(tt_t1)
print(f"  tx-tx edges (mapped to tx idx): {n_tt:,}  all within-timestep ✓")

# Each virtual wallet pairs T1 and T2 of one tx-tx edge.
virtual_w_ids = np.arange(N_W, N_W + n_tt, dtype=np.int64)
row_w_tt_part  = np.concatenate([virtual_w_ids, virtual_w_ids])    # 2 edges per virtual wallet
col_tx_tt_part = np.concatenate([tt_t1,         tt_t2])
e_time_tt_part = np.concatenate([tt_time,       tt_time])

row_w_v2  = np.concatenate([row_w_bp,  row_w_tt_part])
col_tx_v2 = np.concatenate([col_tx_bp, col_tx_tt_part])
e_time_v2 = np.concatenate([e_time_bp, e_time_tt_part])
N_W_v2    = N_W + n_tt
print(f"V2 (bipartite + tx-tx):  {len(row_w_v2):,} edges over {N_W_v2:,} wallets ({n_tt:,} virtual)")

# Sanity: every edge's timestep must be a real tx timestep (1..49)
assert e_time_bp.min() >= 1 and e_time_bp.max() <= N_TIME_STEPS
assert e_time_v2.min() >= 1 and e_time_v2.max() <= N_TIME_STEPS
print("  causality sanity: all edge timestamps in [1, 49] ✓")


V1 (bipartite only):     1,268,260 edges over 822,942 wallets + 203,769 txs
  tx-tx edges (mapped to tx idx): 234,355  all within-timestep ✓
V2 (bipartite + tx-tx):  1,736,970 edges over 1,057,297 wallets (234,355 virtual)
  causality sanity: all edge timestamps in [1, 49] ✓


In [4]:
"""
Multi-Scale Temporal Bipartite Diffusion (MS-TBD)
==================================================

PyTorch implementation of a graph-regularized semi-supervised classifier for
illicit transaction detection on the Elliptic++ bipartite addr↔tx graph,
under STRICT causal evaluation (only edges with τ ≤ t_query are visible).

MATHEMATICAL FORMULATION
------------------------
For each transaction T at time t, we approximately solve the optimization

    min_f   Σ_i w_i · ℓ(f_i, y_i)               (supervised loss on labelled txs at τ < t)
          + μ · f^T L_t^β f                     (smoothness on temporally-decayed bipartite graph)
          + γ · ||f - h_θ(X)||²                 (feature prior — pulls f toward MLP prediction)

via K iterations of the Richardson-style fixed-point update

    f^(k+1) = (1-ω) · h_θ(X)  +  ω · Ã_t^β · f^(k)

decomposed into bipartite half-steps:

    Half-1 (tx → wallet):  g_w     = Σ_{tx∈N(w)} w_e · f_tx     /  Σ_{tx∈N(w)} w_e
    Half-2 (wallet → tx):  f'_tx   = (1-ω)·h_θ(x_tx)
                                    + ω · ( Σ_{w∈N(tx)} w_e · g_w  /  Σ_{w∈N(tx)} w_e )

with edge weights  w_e = 1[τ_e ≤ t] · exp(-β · (t - τ_e)).

MULTI-SCALE: run S parallel diffusions with different learnable β_s. Concatenate
the converged scores as a multi-scale spectral representation.

THIS IS NOT A HEURISTIC. The architecture is the Richardson iteration applied
to a specific optimization problem. Convergence to the unique fixed point is
guaranteed (positive-definite system matrix) at rate O(ρ^K), ρ < 1.

USAGE
-----
    from ms_tbd import MSTBD, build_bipartite_coo, train_mstbd

    row_w, col_tx, e_time = build_bipartite_coo(edges_w, edges_tx, edges_t,
                                                  N_W, N_TX)
    X = torch.tensor(tx_features, dtype=torch.float32)
    y = torch.tensor(tx_labels,   dtype=torch.long)

    model = MSTBD(in_dim=X.shape[1], n_wallets=N_W, n_total_txs=N_TX,
                  n_scales=5, n_iters=8, hidden=64)
    train_mstbd(model, X, y, tx_time, row_w, col_tx, e_time,
                train_end=29, val_end=34, test_start=35,
                n_epochs=30, lr=3e-3, class_weight=8.0)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


# =============================================================================
#  Bipartite operators
# =============================================================================

def build_bipartite_coo(edges_w, edges_tx, edges_t, n_wallets, n_txs):
    """Convert edge lists to (row_w, col_tx, e_time) tensors.

    Each edge connects wallet edges_w[i] to transaction edges_tx[i] which has
    timestep edges_t[i]. The graph is treated as undirected — half-1 of the
    diffusion uses edges as tx→wallet, half-2 as wallet→tx, with the same
    edge list."""
    row_w  = torch.as_tensor(np.asarray(edges_w),  dtype=torch.long)
    col_tx = torch.as_tensor(np.asarray(edges_tx), dtype=torch.long)
    e_time = torch.as_tensor(np.asarray(edges_t),  dtype=torch.long)
    return row_w, col_tx, e_time


def temporal_edge_weights(e_time, t_query, beta):
    """w_e = 1[τ_e ≤ t_query] · exp(-β · (t_query - τ_e))

    Future edges (τ_e > t_query) get HARD ZERO weight to enforce causality.
    """
    delta = (t_query - e_time).to(torch.float32)
    causal = (delta >= 0).to(torch.float32)
    return causal * torch.exp(-beta * torch.clamp(delta, min=0.0))


def diffusion_step(f_tx, h_tx, omega, row_w, col_tx, edge_w, n_wallets, eps=1e-6):
    """One Richardson iteration of the bipartite diffusion.

    Args:
        f_tx     : [N_tx]   current diffused scores
        h_tx     : [N_tx]   feature prior h_θ(X)
        omega    : scalar   interpolation weight ω ∈ (0,1)
        row_w    : [E]      wallet endpoints
        col_tx   : [E]      tx endpoints (in 0..N_tx-1)
        edge_w   : [E]      edge weights (already include causality + decay)
        n_wallets: int

    Returns:
        f_new : [N_tx]   updated scores

    Implementation note: a tx with NO incident causal edges (denom_tx ≈ 0) falls
    back to f_new = h_tx (pure feature prior), avoiding the 0/0 instability. We
    detect this via `has_support = denom_tx > eps` and use a soft mask.
    """
    n_tx = f_tx.shape[0]
    device, dtype = f_tx.device, f_tx.dtype

    # Half-1: tx → wallet
    f_per_edge = f_tx.index_select(0, col_tx)
    num_w   = torch.zeros(n_wallets, device=device, dtype=dtype)
    denom_w = torch.zeros(n_wallets, device=device, dtype=dtype)
    num_w.scatter_add_(0, row_w, edge_w * f_per_edge)
    denom_w.scatter_add_(0, row_w, edge_w)
    g = num_w / (denom_w + eps)

    # Half-2: wallet → tx
    g_per_edge = g.index_select(0, row_w)
    num_tx   = torch.zeros(n_tx, device=device, dtype=dtype)
    denom_tx = torch.zeros(n_tx, device=device, dtype=dtype)
    num_tx.scatter_add_(0, col_tx, edge_w * g_per_edge)
    denom_tx.scatter_add_(0, col_tx, edge_w)
    diffused = num_tx / (denom_tx + eps)

    # Interpolate: where the tx has no causal support, fall back to h_tx
    has_support = (denom_tx > eps).to(dtype)
    return h_tx * (1.0 - omega * has_support) + omega * has_support * diffused


# =============================================================================
#  Model
# =============================================================================

class FeatureMLP(nn.Module):
    def __init__(self, in_dim, hidden=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MSTBD(nn.Module):
    """Multi-Scale Temporal Bipartite Diffusion classifier."""

    def __init__(self, in_dim, n_wallets, n_total_txs,
                 n_scales=5, n_iters=8, hidden=64, dropout=0.2,
                 beta_init=None, omega_init=0.5):
        super().__init__()
        self.n_wallets = n_wallets
        self.n_total_txs = n_total_txs
        self.S = n_scales
        self.K = n_iters

        self.feature_mlp = FeatureMLP(in_dim, hidden=hidden, dropout=dropout)

        if beta_init is None:
            beta_init = np.geomspace(0.05, 2.0, n_scales).astype(np.float32)
        self.beta_log    = nn.Parameter(torch.log(torch.tensor(beta_init, dtype=torch.float32)))
        self.omega_logit = nn.Parameter(torch.tensor(np.log(omega_init/(1-omega_init)),
                                                       dtype=torch.float32))

        head_in = in_dim + 2 * n_scales
        self.head = nn.Sequential(
            nn.Linear(head_in, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 2),
        )

    @property
    def betas(self): return torch.exp(self.beta_log)
    @property
    def omega(self): return torch.sigmoid(self.omega_logit)

    def diffuse_at_t(self, t_query, x_all, row_w, col_tx, e_time):
        """Run K iterations of the multi-scale diffusion at query timestep t_query.

        Returns:
            scores  : [N_tx, S]  per-tx, per-scale converged diffused scores
            support : [N_tx, S]  per-tx, per-scale total causal incident edge mass
                                 (tanh-squashed, used as a "trust the graph" indicator)
        """
        device = x_all.device
        h_tx = self.feature_mlp(x_all)
        omega = self.omega

        scores, support = [], []
        for s in range(self.S):
            beta_s = self.betas[s]
            edge_w = temporal_edge_weights(e_time, t_query, beta_s)
            f = h_tx.clone()
            for _ in range(self.K):
                f = diffusion_step(f, h_tx, omega, row_w, col_tx, edge_w, self.n_wallets)
            scores.append(f)
            denom_tx = torch.zeros(self.n_total_txs, device=device, dtype=f.dtype)
            denom_tx.scatter_add_(0, col_tx, edge_w)
            support.append(torch.tanh(denom_tx))
        return torch.stack(scores, dim=1), torch.stack(support, dim=1)

    def forward(self, t_query, target_idx, x_all, row_w, col_tx, e_time):
        scores, support = self.diffuse_at_t(t_query, x_all, row_w, col_tx, e_time)
        x_t = x_all.index_select(0, target_idx)
        s_t = scores.index_select(0, target_idx)
        u_t = support.index_select(0, target_idx)
        return self.head(torch.cat([x_t, s_t, u_t], dim=-1))


# =============================================================================
#  Training utility
# =============================================================================

def train_mstbd(
    model, X, y, tx_time, row_w, col_tx, e_time,
    train_end, val_end, test_start,
    n_epochs=30, lr=3e-3, class_weight=8.0,
    grad_clip=5.0, device="cpu", verbose=True,
):
    """Train MS-TBD with chronological per-timestep batching.

    Args:
        X         : [N_tx, F]  raw tx features (already standardised)
        y         : [N_tx]     labels (0=licit, 1=illicit, -1=unknown)
        tx_time   : [N_tx]     int timestep per tx
        row_w/col_tx/e_time : bipartite edge tensors
        train_end : last timestep for training (inclusive)
        val_end   : last timestep for validation (inclusive)
        test_start: first timestep for test (inclusive)
        class_weight : weight on the illicit class in cross-entropy

    Returns:
        history : dict of per-epoch metrics
        best_state : state_dict at best val F1
    """
    from sklearn.metrics import f1_score, roc_auc_score

    model = model.to(device)
    X = X.to(device); y = y.to(device)
    row_w = row_w.to(device); col_tx = col_tx.to(device); e_time = e_time.to(device)
    tx_time_np = tx_time.cpu().numpy() if torch.is_tensor(tx_time) else np.asarray(tx_time)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    cw  = torch.tensor([1.0, class_weight], dtype=torch.float32, device=device)

    train_ts = sorted({int(t) for t in tx_time_np if t <= train_end})
    val_ts   = sorted({int(t) for t in tx_time_np if train_end < t <= val_end})
    test_ts  = sorted({int(t) for t in tx_time_np if t >= test_start})

    history = {"train_loss": [], "val_f1": [], "val_auc": [], "test_f1": [], "test_auc": []}
    best_val_f1 = -1.0
    best_state = None

    def eval_split(timesteps):
        model.eval()
        all_y, all_p, all_score = [], [], []
        with torch.no_grad():
            for t in timesteps:
                tgt = np.where((tx_time_np == t) & (y.cpu().numpy() != -1))[0]
                if len(tgt) == 0: continue
                ti = torch.tensor(tgt, dtype=torch.long, device=device)
                logits = model(t, ti, X, row_w, col_tx, e_time)
                prob = F.softmax(logits, -1)[:, 1].cpu().numpy()
                pred = (prob > 0.5).astype(np.int64)
                all_y.extend(y[ti].cpu().numpy().tolist())
                all_p.extend(pred.tolist())
                all_score.extend(prob.tolist())
        if len(all_y) == 0: return float("nan"), float("nan")
        f1 = f1_score(all_y, all_p, pos_label=1, zero_division=0)
        try:
            auc = roc_auc_score(all_y, all_score)
        except ValueError:
            auc = float("nan")
        return f1, auc

    for epoch in range(n_epochs):
        model.train()
        epoch_losses = []
        np.random.shuffle(train_ts)
        for t in train_ts:
            tgt = np.where((tx_time_np == t) & (y.cpu().numpy() != -1))[0]
            if len(tgt) == 0: continue
            ti = torch.tensor(tgt, dtype=torch.long, device=device)
            opt.zero_grad()
            logits = model(t, ti, X, row_w, col_tx, e_time)
            loss = F.cross_entropy(logits, y[ti], weight=cw)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
            epoch_losses.append(loss.item())

        train_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
        val_f1, val_auc = eval_split(val_ts)
        test_f1, test_auc = eval_split(test_ts)

        history["train_loss"].append(train_loss)
        history["val_f1"].append(val_f1); history["val_auc"].append(val_auc)
        history["test_f1"].append(test_f1); history["test_auc"].append(test_auc)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if verbose:
            print(f"  ep{epoch:2d}  loss={train_loss:.4f}  "
                  f"val_F1={val_f1:.4f}  val_AUC={val_auc:.4f}  "
                  f"test_F1={test_f1:.4f}  test_AUC={test_auc:.4f}  "
                  f"βs={model.betas.detach().cpu().numpy().round(3)}  "
                  f"ω={model.omega.item():.3f}")

    return history, best_state


# =============================================================================
#  Smoke tests (quick — run by `python ms_tbd_final.py`)
# =============================================================================

def _smoke_test():
    torch.manual_seed(0); np.random.seed(0)
    N_W, N_TX, F_DIM, N_T = 100, 200, 8, 10
    edges_w = np.random.randint(0, N_W, 400)
    edges_tx = np.random.randint(0, N_TX, 400)
    edges_t = np.random.randint(0, N_T, 400)
    row_w, col_tx, e_time = build_bipartite_coo(edges_w, edges_tx, edges_t, N_W, N_TX)

    tx_time = torch.tensor(np.random.randint(0, N_T, N_TX))
    y = torch.tensor(np.random.randint(0, 2, N_TX))
    y[np.random.choice(N_TX, 30, replace=False)] = -1
    X = torch.randn(N_TX, F_DIM)

    model = MSTBD(F_DIM, N_W, N_TX, n_scales=3, n_iters=4, hidden=16)
    hist, best = train_mstbd(
        model, X, y, tx_time, row_w, col_tx, e_time,
        train_end=5, val_end=7, test_start=8,
        n_epochs=3, lr=1e-2, class_weight=2.0, verbose=True,
    )
    print("\nSmoke test passed.")


if __name__ == "__main__":
    _smoke_test()

  ep 0  loss=0.6500  val_F1=0.4865  val_AUC=0.4503  test_F1=0.8649  test_AUC=0.4375  βs=[0.049 0.327 2.035]  ω=0.500
  ep 1  loss=0.6150  val_F1=0.4865  val_AUC=0.5205  test_F1=0.8649  test_AUC=0.3875  βs=[0.048 0.335 2.095]  ω=0.496
  ep 2  loss=0.6061  val_F1=0.4865  val_AUC=0.5322  test_F1=0.8649  test_AUC=0.4000  βs=[0.047 0.342 2.166]  ω=0.493

Smoke test passed.


## Prepare inputs and run

Standardise tx features (fit on **train rows only**, `t ≤ 29`), build torch tensors, define the train/val/test splits, then train both V1 and V2.


In [5]:
# Standardise tx features — fit on TRAIN-ONLY rows so the scaler doesn't leak future stats.
labelled    = (tx_label != -1)
neural_train_mask = labelled & (tx_time_np <= TRAIN_END)
neural_val_mask   = labelled & (tx_time_np >  TRAIN_END) & (tx_time_np <= VAL_END)
neural_test_mask  = labelled & (tx_time_np >= TEST_START)

# RF baseline uses the phase-1 split (train t <= 34, no carved val)
rf_train_mask = labelled & (tx_time_np <= RF_TRAIN_END)
rf_test_mask  = labelled & (tx_time_np >  RF_TRAIN_END)

print(f"MS-TBD train (t<= {TRAIN_END}):     n={int(neural_train_mask.sum()):,}  illicit_rate={tx_label[neural_train_mask].mean():.4f}")
print(f"MS-TBD val   ({TRAIN_END}<t<={VAL_END}):   n={int(neural_val_mask.sum()):,}  illicit_rate={tx_label[neural_val_mask].mean():.4f}")
print(f"MS-TBD test  (t>= {TEST_START}):    n={int(neural_test_mask.sum()):,}  illicit_rate={tx_label[neural_test_mask].mean():.4f}")
print(f"RF      train (t<= {RF_TRAIN_END}):     n={int(rf_train_mask.sum()):,}")
print(f"RF      test  (t> {RF_TRAIN_END}):     n={int(rf_test_mask.sum()):,}")

# Standardiser fit on the MS-TBD training rows only (t <= 29)
scaler = StandardScaler().fit(tx_X_raw[neural_train_mask])
tx_X_std = scaler.transform(tx_X_raw).astype(np.float32)

# Torch tensors for MS-TBD
X_t       = torch.from_numpy(tx_X_std)                                      # [N_TX, F_TX]
y_t       = torch.from_numpy(tx_label.astype(np.int64))                    # [N_TX]
tx_time_t = torch.from_numpy(tx_time_np.astype(np.int64))                  # [N_TX]
print(f"\nX (standardised): {tuple(X_t.shape)}  y: {tuple(y_t.shape)}")


MS-TBD train (t<= 29):     n=26,381  illicit_rate=0.1088
MS-TBD val   (29<t<=34):   n=3,513  illicit_rate=0.1682
MS-TBD test  (t>= 35):    n=16,670  illicit_rate=0.0650
RF      train (t<= 34):     n=29,894
RF      test  (t> 34):     n=16,670

X (standardised): (203769, 182)  y: (203769,)


In [6]:
# RandomForest baseline on raw 182 features (matches phase 1 / 1c exactly)
print("=== Baseline: RF[raw 182 features], train t<=34 / test t>=35 ===")
y_train_rf = tx_label[rf_train_mask]
y_test_rf  = tx_label[rf_test_mask]
test_t_rf  = tx_time_np[rf_test_mask]

t0 = time.time()
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                            n_jobs=-1, random_state=RANDOM_SEED)
rf.fit(tx_X_raw[rf_train_mask], y_train_rf)
yp_rf  = rf.predict(tx_X_raw[rf_test_mask])
ypr_rf = rf.predict_proba(tx_X_raw[rf_test_mask])[:, 1]
f1_rf    = f1_score(y_test_rf, yp_rf, pos_label=1, zero_division=0)
auc_rf   = roc_auc_score(y_test_rf, ypr_rf)
prauc_rf = average_precision_score(y_test_rf, ypr_rf)
print(f"  trained in {time.time()-t0:.1f}s")
print(f"  F1={f1_rf:.4f}  AUC={auc_rf:.4f}  PR-AUC={prauc_rf:.4f}")


=== Baseline: RF[raw 182 features], train t<=34 / test t>=35 ===
  trained in 3.6s
  F1=0.8026  AUC=0.9245  PR-AUC=0.7927


### MS-TBD V1 — bipartite addr↔tx edges only

In [7]:
print("=== MS-TBD V1: bipartite addr<->tx edges only ===")
torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

row_w_v1_t, col_tx_v1_t, e_time_v1_t = build_bipartite_coo(row_w_bp, col_tx_bp, e_time_bp,
                                                             N_W, N_TX)
model_v1 = MSTBD(in_dim=F_TX, n_wallets=N_W, n_total_txs=N_TX,
                 n_scales=N_SCALES, n_iters=N_ITERS, hidden=HIDDEN, dropout=DROPOUT)
print(f"  model params (V1): {sum(p.numel() for p in model_v1.parameters()):,}")

t0 = time.time()
hist_v1, best_state_v1 = train_mstbd(
    model_v1, X_t, y_t, tx_time_t,
    row_w_v1_t, col_tx_v1_t, e_time_v1_t,
    train_end=TRAIN_END, val_end=VAL_END, test_start=TEST_START,
    n_epochs=N_EPOCHS, lr=LR, class_weight=CLASS_WEIGHT_NEURAL,
    grad_clip=GRAD_CLIP, device=str(DEVICE), verbose=True,
)
print(f"\nV1 trained in {time.time()-t0:.1f}s")

# Restore best-val checkpoint and compute final test metrics
if best_state_v1 is not None:
    model_v1.load_state_dict(best_state_v1)

model_v1.eval()
ty_v1, ypr_v1 = [], []
with torch.no_grad():
    for t in range(TEST_START, N_TIME_STEPS + 1):
        tgt = np.where((tx_time_np == t) & (tx_label != -1))[0]
        if len(tgt) == 0: continue
        ti = torch.tensor(tgt, dtype=torch.long, device=str(DEVICE))
        logits = model_v1(t, ti, X_t.to(DEVICE),
                          row_w_v1_t.to(DEVICE), col_tx_v1_t.to(DEVICE), e_time_v1_t.to(DEVICE))
        prob = F.softmax(logits, -1)[:, 1].cpu().numpy()
        ty_v1.append(tx_label[tgt]); ypr_v1.append(prob)
ty_v1  = np.concatenate(ty_v1)
ypr_v1 = np.concatenate(ypr_v1)
yp_v1  = (ypr_v1 >= 0.5).astype(np.int64)
f1_v1    = f1_score(ty_v1, yp_v1, pos_label=1, zero_division=0)
auc_v1   = roc_auc_score(ty_v1, ypr_v1)
prauc_v1 = average_precision_score(ty_v1, ypr_v1)
print(f"\n[V1: MS-TBD bipartite]  F1={f1_v1:.4f}  AUC={auc_v1:.4f}  PR-AUC={prauc_v1:.4f}")


=== MS-TBD V1: bipartite addr<->tx edges only ===
  model params (V1): 28,425
  ep 0  loss=0.6322  val_F1=0.4052  val_AUC=0.7860  test_F1=0.2493  test_AUC=0.8219  βs=[0.05  0.126 0.315 0.794 1.995]  ω=0.501
  ep 1  loss=0.5046  val_F1=0.4612  val_AUC=0.8484  test_F1=0.2649  test_AUC=0.8632  βs=[0.05  0.125 0.313 0.788 1.982]  ω=0.502
  ep 2  loss=0.4131  val_F1=0.4987  val_AUC=0.8768  test_F1=0.2963  test_AUC=0.8772  βs=[0.049 0.124 0.31  0.781 1.967]  ω=0.505
  ep 3  loss=0.3259  val_F1=0.5438  val_AUC=0.9005  test_F1=0.3078  test_AUC=0.8801  βs=[0.049 0.123 0.307 0.773 1.952]  ω=0.506
  ep 4  loss=0.2703  val_F1=0.5477  val_AUC=0.9064  test_F1=0.3080  test_AUC=0.8900  βs=[0.048 0.122 0.304 0.766 1.936]  ω=0.508
  ep 5  loss=0.2382  val_F1=0.5886  val_AUC=0.9235  test_F1=0.3351  test_AUC=0.8945  βs=[0.048 0.121 0.302 0.761 1.922]  ω=0.509
  ep 6  loss=0.2100  val_F1=0.5844  val_AUC=0.9242  test_F1=0.3316  test_AUC=0.9009  βs=[0.048 0.12  0.3   0.756 1.91 ]  ω=0.510
  ep 7  loss=0.1949

### MS-TBD V2 — bipartite addr↔tx + tx-tx via virtual wallets

In [8]:
print("=== MS-TBD V2: bipartite + tx-tx via virtual wallets ===")
torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

row_w_v2_t, col_tx_v2_t, e_time_v2_t = build_bipartite_coo(row_w_v2, col_tx_v2, e_time_v2,
                                                             N_W_v2, N_TX)
model_v2 = MSTBD(in_dim=F_TX, n_wallets=N_W_v2, n_total_txs=N_TX,
                 n_scales=N_SCALES, n_iters=N_ITERS, hidden=HIDDEN, dropout=DROPOUT)
print(f"  model params (V2): {sum(p.numel() for p in model_v2.parameters()):,}")

t0 = time.time()
hist_v2, best_state_v2 = train_mstbd(
    model_v2, X_t, y_t, tx_time_t,
    row_w_v2_t, col_tx_v2_t, e_time_v2_t,
    train_end=TRAIN_END, val_end=VAL_END, test_start=TEST_START,
    n_epochs=N_EPOCHS, lr=LR, class_weight=CLASS_WEIGHT_NEURAL,
    grad_clip=GRAD_CLIP, device=str(DEVICE), verbose=True,
)
print(f"\nV2 trained in {time.time()-t0:.1f}s")

if best_state_v2 is not None:
    model_v2.load_state_dict(best_state_v2)

model_v2.eval()
ty_v2, ypr_v2 = [], []
with torch.no_grad():
    for t in range(TEST_START, N_TIME_STEPS + 1):
        tgt = np.where((tx_time_np == t) & (tx_label != -1))[0]
        if len(tgt) == 0: continue
        ti = torch.tensor(tgt, dtype=torch.long, device=str(DEVICE))
        logits = model_v2(t, ti, X_t.to(DEVICE),
                          row_w_v2_t.to(DEVICE), col_tx_v2_t.to(DEVICE), e_time_v2_t.to(DEVICE))
        prob = F.softmax(logits, -1)[:, 1].cpu().numpy()
        ty_v2.append(tx_label[tgt]); ypr_v2.append(prob)
ty_v2  = np.concatenate(ty_v2)
ypr_v2 = np.concatenate(ypr_v2)
yp_v2  = (ypr_v2 >= 0.5).astype(np.int64)
f1_v2    = f1_score(ty_v2, yp_v2, pos_label=1, zero_division=0)
auc_v2   = roc_auc_score(ty_v2, ypr_v2)
prauc_v2 = average_precision_score(ty_v2, ypr_v2)
print(f"\n[V2: MS-TBD bipartite + tx-tx]  F1={f1_v2:.4f}  AUC={auc_v2:.4f}  PR-AUC={prauc_v2:.4f}")


=== MS-TBD V2: bipartite + tx-tx via virtual wallets ===
  model params (V2): 28,425
  ep 0  loss=0.6323  val_F1=0.4048  val_AUC=0.7851  test_F1=0.2492  test_AUC=0.8216  βs=[0.05  0.126 0.315 0.794 1.994]  ω=0.501
  ep 1  loss=0.5057  val_F1=0.4603  val_AUC=0.8472  test_F1=0.2644  test_AUC=0.8621  βs=[0.05  0.125 0.313 0.788 1.982]  ω=0.502
  ep 2  loss=0.4163  val_F1=0.4972  val_AUC=0.8755  test_F1=0.2949  test_AUC=0.8748  βs=[0.049 0.124 0.31  0.781 1.968]  ω=0.504
  ep 3  loss=0.3309  val_F1=0.5314  val_AUC=0.8984  test_F1=0.3010  test_AUC=0.8767  βs=[0.049 0.123 0.308 0.774 1.952]  ω=0.505
  ep 4  loss=0.2755  val_F1=0.5343  val_AUC=0.9036  test_F1=0.3004  test_AUC=0.8860  βs=[0.048 0.122 0.305 0.766 1.935]  ω=0.506
  ep 5  loss=0.2430  val_F1=0.5768  val_AUC=0.9213  test_F1=0.3239  test_AUC=0.8912  βs=[0.048 0.121 0.302 0.76  1.92 ]  ω=0.507
  ep 6  loss=0.2139  val_F1=0.5710  val_AUC=0.9215  test_F1=0.3187  test_AUC=0.8979  βs=[0.048 0.12  0.3   0.754 1.905]  ω=0.507
  ep 7  loss

In [10]:
# === Per-timestep test breakdown ===
print("=" * 80)
print("Per-timestep test F1 (illicit class)")
print("=" * 80)
print(f"{'t':>3}  {'n':>5}  {'illicit':>7}  {'RF':>8}  {'V1':>8}  {'V2':>8}  {'V2-V1':>7}  {'V2-RF':>7}")

# Build per-timestep masks aligned to V1/V2 prediction arrays.
v1_t = []; v2_t = []
for t in range(TEST_START, N_TIME_STEPS + 1):
    tgt = np.where((tx_time_np == t) & (tx_label != -1))[0]
    v1_t.extend([t] * len(tgt))
    v2_t.extend([t] * len(tgt))
v1_t = np.array(v1_t); v2_t = np.array(v2_t)
assert len(v1_t) == len(ty_v1) == len(v2_t) == len(ty_v2)

for t in range(TEST_START, N_TIME_STEPS + 1):
    m_rf = (test_t_rf == t)
    if not m_rf.any(): continue
    yt_rf = y_test_rf[m_rf]
    if yt_rf.sum() == 0:
        print(f"{t:>3}  {int(m_rf.sum()):>5}  {int(yt_rf.sum()):>7}  "
              + "  ".join(f"{'NaN':>8}" for _ in range(3))
              + "  " + "  ".join(f"{'NaN':>7}" for _ in range(2)))
        continue
    f1_rf_t = f1_score(yt_rf, yp_rf[m_rf], pos_label=1, zero_division=0)
    m_v1 = (v1_t == t); m_v2 = (v2_t == t)
    f1_v1_t = f1_score(ty_v1[m_v1], yp_v1[m_v1], pos_label=1, zero_division=0) if m_v1.any() else float("nan")
    f1_v2_t = f1_score(ty_v2[m_v2], yp_v2[m_v2], pos_label=1, zero_division=0) if m_v2.any() else float("nan")
    print(f"{t:>3}  {int(m_rf.sum()):>5}  {int(yt_rf.sum()):>7}  "
          f"{f1_rf_t:>8.4f}  {f1_v1_t:>8.4f}  {f1_v2_t:>8.4f}  "
          f"{f1_v2_t - f1_v1_t:>+7.3f}  {f1_v2_t - f1_rf_t:>+7.3f}")

# === Summary ===
print("\n" + "=" * 80)
print(f"{'Model':35s}  {'F1':>8s}  {'AUC':>8s}  {'PR-AUC':>8s}")
print("=" * 80)
results = {
    "RF[raw 182] (train t<=34)":        (f1_rf, auc_rf, prauc_rf),
    "MS-TBD V1: bipartite":              (f1_v1, auc_v1, prauc_v1),
    "MS-TBD V2: bipartite + tx-tx":      (f1_v2, auc_v2, prauc_v2),
}
for name, (f1, auc, prauc) in sorted(results.items(), key=lambda kv: -kv[1][0]):
    print(f"  {name:33s}  {f1:>8.4f}  {auc:>8.4f}  {prauc:>8.4f}")

print()
print(f"Δ V1 vs RF (does the bipartite diffusion help?):       F1={f1_v1-f1_rf:+.4f}  PR-AUC={prauc_v1-prauc_rf:+.4f}")
print(f"Δ V2 vs V1 (does adding tx-tx edges help?):            F1={f1_v2-f1_v1:+.4f}  PR-AUC={prauc_v2-prauc_v1:+.4f}")
print(f"Δ V2 vs RF (full system vs RF baseline):               F1={f1_v2-f1_rf:+.4f}  PR-AUC={prauc_v2-prauc_rf:+.4f}")

print("\nNotes:")
print(f"  - Causality enforced: edge weights w_e = 1[τ_e ≤ t] · exp(-β · (t - τ_e)).")
print(f"  - V2 encodes each tx-tx edge as 2 bipartite edges through a unique virtual wallet,")
print(f"    so the same MS-TBD operator handles both versions.")
print(f"  - No AddrAddr edges (no timestamps -> would leak future structure under causal protocol).")
print(f"  - MS-TBD: train t<= {TRAIN_END}, val ({TRAIN_END}, {VAL_END}], test t>= {TEST_START}.")
print(f"           {N_SCALES} scales, {N_ITERS} iters, hidden={HIDDEN}, lr={LR}, class_weight={CLASS_WEIGHT_NEURAL}.")
print(f"  - RF baseline: train t<= {RF_TRAIN_END} (full phase-1 protocol), no val.")


Per-timestep test F1 (illicit class)
  t      n  illicit        RF        V1        V2    V2-V1    V2-RF
 35   1341      182    0.9805    0.9105    0.8789   -0.032   -0.102
 36   1708       33    0.8814    0.2949    0.2876   -0.007   -0.594
 37    498       40    0.7500    0.4673    0.4717   +0.004   -0.278
 38    756      111    0.9429    0.5477    0.5031   -0.045   -0.440
 39   1183       81    0.9054    0.5161    0.4364   -0.080   -0.469
 40   1211      112    0.7416    0.4876    0.4456   -0.042   -0.296
 41   1132      116    0.8952    0.3930    0.3717   -0.021   -0.524
 42   2154      239    0.8489    0.7490    0.7116   -0.037   -0.137
 43   1370       24    0.0000    0.0000    0.0000   +0.000   +0.000
 44   1591       24    0.0690    0.0364    0.0377   +0.001   -0.031
 45   1221        5    0.0000    0.0455    0.0526   +0.007   +0.053
 46    712        2    0.5000    0.0000    0.0000   +0.000   -0.500
 47    846       22    0.0000    0.0000    0.0000   +0.000   +0.000
 48    471 